In [3]:
import logging
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Set up logging per project standards
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# -------------------------------------------------------------------------
# STEP 1: Load Data & Create Proxy Target
# -------------------------------------------------------------------------
df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")
logger.info(f"Loaded dataset with shape: {df.shape}")

# Parse datetime
df["date_time"] = pd.to_datetime(df["date_time"])

# Define weather categories
LOW_VISIBILITY_WEATHER = ["Mist", "Fog", "Haze", "Smoke"]
SEVERE_WEATHER = [
    "Squall",
    "Proximity Thunderstorm",
    "Thunderstorm",
    "Heavy Snow",
    "Heavy Rain",
] + LOW_VISIBILITY_WEATHER

df["is_low_visibility"] = (
    df["weather_main"].isin(LOW_VISIBILITY_WEATHER).astype(int)
)

# Derive quartile congestion categories and proxy label
q1, q2, q3 = df["traffic_volume"].quantile([0.25, 0.5, 0.75]).values


def bucket(v):
    if v <= q1:
        return "Low"
    elif v <= q2:
        return "Medium"
    elif v <= q3:
        return "High"
    return "Severe"


df["congestion_category"] = df["traffic_volume"].apply(bucket)

high_congestion = df["congestion_category"].isin(["High", "Severe"])
risky_weather = df["weather_main"].isin(SEVERE_WEATHER) | (
    df["is_low_visibility"] == 1
)
df["high_risk"] = (high_congestion & risky_weather).astype(int)

# -------------------------------------------------------------------------
# STEP 2: Feature Engineering
# -------------------------------------------------------------------------
# Time-based features
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Holiday flag
df["is_holiday"] = (
    df["holiday"].notna() & (df["holiday"] != "None")
).astype(int)

# Cyclical encodings for hour and day of week
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)
df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7.0)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7.0)

# One-hot encoding for weather categories
weather_dummies = pd.get_dummies(
    df["weather_main"], prefix="weather", drop_first=True
)
df = pd.concat([df, weather_dummies], axis=1)

# Assemble final feature list
feature_cols = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
] + list(weather_dummies.columns)

X = df[feature_cols]
y_class = df["high_risk"]
y_reg = df["traffic_volume"]

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train_c, y_test_c, y_train_r, y_test_r = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

# Standardize feature matrices
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logger.info("Feature engineering and data preparation completed.")

# -------------------------------------------------------------------------
# STEP 3: Model Training
# -------------------------------------------------------------------------
# Classification Models
clf_lr = LogisticRegression(max_iter=1000, random_state=42)
clf_lr.fit(X_train_scaled, y_train_c)
y_pred_lr = clf_lr.predict(X_test_scaled)
y_prob_lr = clf_lr.predict_proba(X_test_scaled)[:, 1]

clf_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_rf.fit(X_train, y_train_c)
y_pred_rf = clf_rf.predict(X_test)
y_prob_rf = clf_rf.predict_proba(X_test)[:, 1]

# Regression Models
reg_ridge = Ridge(alpha=1.0)
reg_ridge.fit(X_train_scaled, y_train_r)
y_pred_ridge = reg_ridge.predict(X_test_scaled)

reg_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
reg_rf.fit(X_train, y_train_r)
y_pred_reg_rf = reg_rf.predict(X_test)

# -------------------------------------------------------------------------
# STEP 4: Print Evaluation Summary
# -------------------------------------------------------------------------
print("\n" + "=" * 60)
print("CLASSIFICATION METRICS COMPARISON (Target: high_risk)")
print("=" * 60)
clf_metrics = pd.DataFrame(
    {
        "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC AUC"],
        "Logistic Regression": [
            accuracy_score(y_test_c, y_pred_lr),
            precision_score(y_test_c, y_pred_lr, zero_division=0),
            recall_score(y_test_c, y_pred_lr),
            f1_score(y_test_c, y_pred_lr),
            roc_auc_score(y_test_c, y_prob_lr),
        ],
        "Random Forest": [
            accuracy_score(y_test_c, y_pred_rf),
            precision_score(y_test_c, y_pred_rf, zero_division=0),
            recall_score(y_test_c, y_pred_rf),
            f1_score(y_test_c, y_pred_rf),
            roc_auc_score(y_test_c, y_prob_rf),
        ],
    }
)
print(clf_metrics.to_string(index=False))

print("\n" + "=" * 60)
print("REGRESSION METRICS COMPARISON (Target: traffic_volume)")
print("=" * 60)
reg_metrics = pd.DataFrame(
    {
        "Metric": ["MAE (vehicles/hr)", "R-squared (R²)"],
        "Ridge Regression": [
            mean_absolute_error(y_test_r, y_pred_ridge),
            r2_score(y_test_r, y_pred_ridge),
        ],
        "Random Forest": [
            mean_absolute_error(y_test_r, y_pred_reg_rf),
            r2_score(y_test_r, y_pred_reg_rf),
        ],
    }
)
print(reg_metrics.to_string(index=False))

2026-10-05 17:14:21,769 - INFO - Loaded dataset with shape: (48204, 9)
2026-10-05 17:14:22,000 - INFO - Feature engineering and data preparation completed.



CLASSIFICATION METRICS COMPARISON (Target: high_risk)
   Metric  Logistic Regression  Random Forest
 Accuracy             0.987346       0.993258
Precision             0.925700       0.959854
   Recall             0.925700       0.961023
 F1-Score             0.925700       0.960438
  ROC AUC             0.997353       0.998631

REGRESSION METRICS COMPARISON (Target: traffic_volume)
           Metric  Ridge Regression  Random Forest
MAE (vehicles/hr)        814.668128     254.205156
   R-squared (R²)          0.725637       0.945882
